In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, recall_score

# Load the Titanic dataset
df = pd.read_csv('Titanic-Dataset.csv')
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [3]:
# Handle missing values, apply categorical encoding, and drop unnecessary columns
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Sex label encoding: male=1, female=0
df['Sex'] = (df['Sex'] == 'male').astype(int)
# Embarked label encoding: C=0, Q=1, S=2
le = LabelEncoder()
df['Embarked'] = le.fit_transform(df['Embarked'].astype(str))

# Features and target
feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[feature_cols]
y = df['Survived']

# Scale features for SVM and Logistic Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=X.index)

In [4]:
# Split into train and test sets (same split for both models)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.1, random_state=42
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 801, Test size: 90


In [5]:
# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr, zero_division=0)

print("Logistic Regression")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"Recall: {lr_recall:.4f}")

Logistic Regression
Accuracy: 0.8444
Recall: 0.8333


In [8]:
# --- SVM ---
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm, zero_division=0)

print("SVM")
print(f"Accuracy: {svm_accuracy:.4f}")
print(f"Recall: {svm_recall:.4f}")

SVM
Accuracy: 0.8111
Recall: 0.7778


In [9]:
# Compare performance
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM'],
    'Accuracy': [lr_accuracy, svm_accuracy],
    'Recall': [lr_recall, svm_recall]
})
print("Result comparison:")
print(comparison.to_string(index=False))

Result comparison:
              Model  Accuracy   Recall
Logistic Regression  0.733333 0.583333
                SVM  0.722222 0.555556


In [10]:
# Create a DataFrame with three passengers and predict survival
# Use same feature columns; values must be scaled with the same scaler
passengers = pd.DataFrame({
    'Pclass': [1, 3, 2],
    'Sex': [0, 1, 0],           # 0=female, 1=male
    'Age': [30, 25, 8],
    'SibSp': [0, 0, 2],
    'Parch': [0, 0, 2],
    'Fare': [50.0, 7.5, 25.0],
    'Embarked': [2, 2, 0]       # C=0, Q=1, S=2 (match LabelEncoder order)
})

passengers_scaled = scaler.transform(passengers)
passengers_scaled = pd.DataFrame(passengers_scaled, columns=feature_cols)

# Predict with both models
pred_lr = lr_model.predict(passengers_scaled)
pred_svm = svm_model.predict(passengers_scaled)

passengers['Pred_LR'] = pred_lr
passengers['Pred_SVM'] = pred_svm
passengers['LR_Result'] = ['survived' if p == 1 else 'not survived' for p in pred_lr]
passengers['SVM_Result'] = ['survived' if p == 1 else 'not survived' for p in pred_svm]

print("Prediction results:")
print(passengers[['Pclass', 'Sex', 'Age', 'Fare', 'LR_Result', 'SVM_Result']].to_string(index=False))
print()
for i in range(3):
    print(f"Passenger {i+1}: Logistic Regression: {passengers['LR_Result'].iloc[i]}, SVM: {passengers['SVM_Result'].iloc[i]}")

Prediction results:
 Pclass  Sex  Age  Fare    LR_Result   SVM_Result
      1    0   30  50.0     survived     survived
      3    1   25   7.5 not survived not survived
      2    0    8  25.0     survived     survived

Passenger 1: Logistic Regression: survived, SVM: survived
Passenger 2: Logistic Regression: not survived, SVM: not survived
Passenger 3: Logistic Regression: survived, SVM: survived
